# Analisis Determinan NEET dengan Regresi Logistik Multinomial

Notebook ini berisi alur kerja untuk menganalisis faktor-faktor penentu status NEET pada level individu menggunakan data mikro Sakernas. Tujuannya adalah untuk membedakan determinan antara **NEET Penganggur** dan **NEET Inaktif**.

## 1. Persiapan Data dan Pustaka

Memuat pustaka yang diperlukan dan memuat data mikro Sakernas (diasumsikan sudah dalam format CSV). Sel ini juga akan menunjukkan cara membuat variabel dependen kategorikal (STATUS_NEET).

In [ ]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# --- PENTING: Ganti 'Data/sakernas_micro.csv' dengan path file data Sakernas Anda ---
file_path = 'Data/sakernas_micro.csv' 

try:
    # Memuat data mikro
    df = pd.read_csv(file_path)
    print("Data mikro Sakernas berhasil dimuat.")

    # --- Membuat Variabel Dependen (Y) ---
    # Ini adalah contoh, nama kolom asli di Sakernas mungkin berbeda
    # Asumsi:
    # 'status_bekerja': 1 jika bekerja, 0 jika tidak
    # 'status_sekolah': 1 jika sekolah, 0 jika tidak
    # 'mencari_pekerjaan': 1 jika aktif mencari kerja, 0 jika tidak
    
    conditions = [
        (df['status_bekerja'] == 1) | (df['status_sekolah'] == 1), # Kondisi untuk Non-NEET
        (df['status_bekerja'] == 0) & (df['status_sekolah'] == 0) & (df['mencari_pekerjaan'] == 1), # Kondisi untuk NEET Penganggur
        (df['status_bekerja'] == 0) & (df['status_sekolah'] == 0) & (df['mencari_pekerjaan'] == 0)  # Kondisi untuk NEET Inaktif
    ]
    
    choices = [0, 1, 2] # 0: Non-NEET, 1: NEET Penganggur, 2: NEET Inaktif
    
    df['STATUS_NEET'] = np.select(conditions, choices, default=np.nan)
    
    # Menghapus data yang tidak relevan (misal: usia di luar 15-24)
    df.dropna(subset=['STATUS_NEET'], inplace=True)
    df['STATUS_NEET'] = df['STATUS_NEET'].astype(int)

    print("\nDistribusi Variabel Dependen:")
    print(df['STATUS_NEET'].value_counts())

except FileNotFoundError:
    print(f"Error: File '{file_path}' tidak ditemukan. Silakan ganti dengan path data Anda.")

## 2. Mendefinisikan Variabel dan Menjalankan Model

Kita akan mendefinisikan variabel independen (prediktor) dan menjalankan model Regresi Logistik Multinomial. Kategori referensi adalah 'Non-NEET' (nilai 0).

In [ ]:
# --- Mendefinisikan Variabel Independen (X) ---
# Ini adalah contoh. Sesuaikan dengan nama kolom di data Sakernas Anda.
# Variabel dummy (kategorikal) perlu dibuat untuk 'jenis_kelamin', 'status_kawin', dll.

# Contoh pembuatan variabel dummy
# df['wanita'] = (df['jenis_kelamin'] == 'Perempuan').astype(int)
# df['menikah'] = (df['status_kawin'] == 'Menikah').astype(int)
# df['lulusan_smk'] = (df['pendidikan_terakhir'] == 'SMK').astype(int)

# Pilih variabel independen untuk model
# X = df[['wanita', 'menikah', 'lulusan_smk', 'usia', 'tinggal_di_kota']]
# X = sm.add_constant(X) # Menambahkan konstanta (intercept)

# Mendefinisikan variabel dependen
# y = df['STATUS_NEET']

# Karena kita tidak punya data asli, kita buat data dummy untuk demonstrasi
np.random.seed(42)
n_samples = 1000
X_demo = pd.DataFrame({
    'const': 1,
    'wanita': np.random.randint(0, 2, n_samples),
    'menikah': np.random.randint(0, 2, n_samples),
    'lulusan_smk': np.random.randint(0, 2, n_samples),
    'usia': np.random.randint(15, 25, n_samples)
})
y_demo = pd.Series(np.random.randint(0, 3, n_samples))

print("--- Menjalankan Model (dengan Data Demo) ---")

# Menjalankan model MNLogit
model = sm.MNLogit(y_demo, X_demo)
result = model.fit()

# Menampilkan ringkasan hasil
print(result.summary())

## 3. Interpretasi Hasil: Odds Ratios

Koefisien dari ringkasan di atas sulit diinterpretasikan secara langsung. Cara terbaik adalah dengan mengubahnya menjadi **Odds Ratios**. Odds ratio menunjukkan seberapa besar perubahan 'peluang' (odds) seorang individu masuk ke dalam satu kategori NEET untuk setiap kenaikan satu unit pada variabel independen.

In [ ]:
# Menghitung dan menampilkan Odds Ratios
odds_ratios = np.exp(result.params)

# Hasil akan dibagi menjadi dua bagian:
# 1. Peluang menjadi NEET Penganggur (kategori 1) vs. Non-NEET
# 2. Peluang menjadi NEET Inaktif (kategori 2) vs. Non-NEET

print("--- Odds Ratios ---")
display(odds_ratios)

print("\n--- Contoh Interpretasi ---")
print("Jika odds ratio untuk 'wanita' pada kategori NEET Inaktif adalah 3.5, artinya:")
print("Seorang wanita memiliki peluang 3.5 kali lebih besar untuk menjadi NEET Inaktif dibandingkan Non-NEET, dengan asumsi variabel lain konstan.")